In [ ]:
# Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split    
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from math import pi
import sqlite3
import tqdm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, LSTM,GlobalAveragePooling1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

In [ ]:
settings_DF=pd.read_csv('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/settings.csv')
# the ID column of settings_DF is unnamed, so we rename it
#settings_DF.drop(columns={'Unnamed: 0'}, inplace=True)
# load signals and merge them, as they have all the same shape
signal_DFs=[]
for signal in ['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']:
#for signal in [ 'vib_spindle', 'AE_spindle']:    
    signal_DFs.append(pd.read_csv('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/'+signal+'.csv'))
    print(signal_DFs[-1].shape)
# merge the signals
merged_DF=pd.concat(signal_DFs, axis=1)
# remove duplicate ID column
merged_DF=merged_DF.loc[:,~merged_DF.columns.duplicated()]
merged_DF.head(10)
#Remove Outliers
settings_DF.drop(settings_DF.loc[settings_DF['cut_no'].isin([17,94])].index, inplace=True)
merged_DF.drop(merged_DF.loc[merged_DF['cut_no'].isin([17,94])].index, inplace=True)

In [ ]:
merged_DF.head(10)

In [ ]:
settings_DF

In [ ]:
# Based on Figure 6 in the DataSheet, we can set a wear treshold to 0.25
wear_threshold = 0.25
# Create a binary classification column based on the wear threshold
settings_DF['wear_class'] = np.where(settings_DF['VB'] > wear_threshold, 1, 0)
# 1 for wear, 0 for no wear

# Merge the settings_DF with the merged_DF on cut_no
binary_class_DF = pd.merge(merged_DF, settings_DF[['cut_no', 'wear_class']], on='cut_no', how='left')
# Quick peek at the binary classification DataFrame
binary_class_DF.head(10)

In [ ]:
# Define the function to reformat data
def reformat_data(selected_signals):
    global sequence_data, labels
    
    # Get the columns to keep
    columns_to_keep = ['cut_no', 'wear_class']
    for signal in selected_signals:
        columns_to_keep.extend([col for col in binary_class_DF.columns if signal in col])
    
    filtered_df = binary_class_DF[columns_to_keep]

    # Group the data by 'cut_no' to represent each time series
    new_sequence_data = []
    new_labels = []

    for cut_no, group in filtered_df.groupby('cut_no'):
        # Drop non-time-series columns and convert to numpy array
        time_series = group.drop(columns=['cut_no', 'wear_class']).to_numpy()
        new_sequence_data.append(time_series)
        new_labels.append(group['wear_class'].iloc[0])

    # Convert to numpy arrays
    sequence_data = np.array(new_sequence_data, dtype=object)
    labels = np.array(new_labels)

    print(f"Using signals: {selected_signals}")
    print(f"Number of sequences: {len(sequence_data)}")
    if len(sequence_data) > 0:
        print(f"Shape of first sequence: {sequence_data[0].shape}")

# Create an interactive widget that executes on user request
interact_manual(reformat_data, selected_signals=widgets.SelectMultiple(
    options=['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle'],
    value=['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle'],
    description='Signals',
    disabled=False
));

In [ ]:
from collections import Counter
# Get the length of each sequence in sequence_data
sequence_lengths = [len(seq) for seq in sequence_data]

# Use Counter to find the frequency of each length
length_counts = Counter(sequence_lengths)

# Check if all sequences have the same length
if len(length_counts) == 1:
    print("All sequences have the same length.")
else:
    print("Sequences have varying lengths.")
    # Optional: Display the distribution of lengths
    print("Length distribution:", length_counts)

# Get the most common length
most_common_length = length_counts.most_common(1)[0][0]

print(f"The most common sequence length is: {most_common_length}")

# Pad or truncate sequences to the most common length
padded_sequences = []
for seq in sequence_data:
    if len(seq) > most_common_length:
        # Truncate
        padded_sequences.append(seq[:most_common_length])
    elif len(seq) < most_common_length:
        # Pad with zeros
        padding = np.zeros((most_common_length - len(seq), seq.shape[1]))
        padded_sequences.append(np.vstack([seq, padding]))
    else:
        padded_sequences.append(seq)

# Update sequence_data with the padded/truncated sequences
sequence_data = np.array(padded_sequences)

print(f"All sequences have been reshaped to length {most_common_length}.")
print(f"New shape of sequence_data: {sequence_data.shape}")

## Model Setup

In [ ]:
# Convert filtered_labels to categorical format for multi-class classification
filtered_labels_categorical = to_categorical(labels)

# Convert filtered_sequences to numpy array for compatibility
filtered_sequences = np.array(sequence_data, dtype=object)

# Split the filtered data into training and testing sets
sequence_train, sequence_test, labels_train, labels_test = train_test_split(
    filtered_sequences, filtered_labels_categorical, test_size=0.2, random_state=42
)

# Convert sequences to 3D shape for LSTM input
sequence_train = np.array([seq for seq in sequence_train])
sequence_test = np.array([seq for seq in sequence_test])
# Ensure the sequences are 3D (samples, timesteps, features)
sequence_train = np.array([seq.reshape(-1, seq.shape[1]) for seq in sequence_train])
sequence_test = np.array([seq.reshape(-1, seq.shape[1]) for seq in sequence_test])
# --- IGNORE ---
# Split the filtered data into training and testing sets

In [ ]:
# Define the LSTM model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(most_common_length, sequence_train[0].shape[1])),
    LSTM(32, return_sequences=False),
    Dense(16, activation='tanh'),
    Dense(filtered_labels_categorical.shape[1], activation='softmax')  # Output layer for classification
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# Model summary
model.summary()

In [ ]:
# Convert sequence_train and sequence_test to proper numpy arrays with float dtype
X_train = np.array(sequence_train, dtype=np.float32)
X_test = np.array(sequence_test, dtype=np.float32)


In [ ]:
# Verify the shape of X_train and labels_train
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of labels_train: {labels_train.shape}")

# Verify the input shape expected by the model
print(f"Expected input shape: {model.input_shape}")

In [ ]:
# Normalize X_train and X_test
X_train = (X_train - np.mean(X_train, axis=0)) / np.std(X_train, axis=0)
X_test = (X_test - np.mean(X_test, axis=0)) / np.std(X_test, axis=0)

In [ ]:
# Train the model
history = model.fit(
    X_train,
    np.array(labels_train),
    epochs=20,
    batch_size=8,
    validation_split=0.2
)

In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(X_test, np.array(labels_test))
# More detailed evaluation
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(labels_test, axis=1)
# Confusion matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()
# Print the evaluation results
print(f"Test Accuracy: {accuracy:.2f}")
print(f"Test Loss: {loss:.2f}")

In [ ]:
#plot loss and accuracy
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()